### IMPLEMENTING RAG PIPELINE

In [1]:
import os
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24360\329546744.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader
e:\MLProjects\RAGlearn\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_community.document_loaders import PyPDFLoader

In [3]:
## FUNCTION TO LOAD ALL THE PDFS FROM THE FOLDER

def process_all_pdfs(pdf_directory):
    all_documents=[]
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")
    for pdf in pdf_files:
        loader = PyPDFLoader(str(pdf))
        loaded_pdf = loader.load()

        for doc in loaded_pdf:
                doc.metadata['source_file'] = pdf.name
                doc.metadata['file_type'] = 'pdf'

        all_documents.extend(loaded_pdf)
        print(f"  ✓ Loaded {len(loaded_pdf)} pages")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

In [4]:
all_pdf_documents = process_all_pdfs("data/")

Found 1 PDF files to process
  ✓ Loaded 7 pages

Total documents loaded: 7


In [5]:
all_pdf_documents

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Md Shamimul Islam; Luis G. Jaimes; Ayesha S. Dina', 'doi': 'https://doi.org/10.48550/arXiv.2604.05458', 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'MA-IDS: Multi-Agent RAG Framework for IoT Network Intrusion Detection with an Experience Library', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2604.05458v1', 'source': 'data\\pdf_files\\MA-IDS-Multi-Agent RAG Framework for IoT Network Intrusion.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'MA-IDS-Multi-Agent RAG Framework for IoT Network Intrusion.pdf', 'file_type': 'pdf'}, page_content='MA-IDS: Multi-Agent RAG Framework for IoT Network Intrusion\nDetection with an Experience Library\nMd Shamimul Islam 1, Luis G. Jaimes 2 and Ayesha S. Dina 3\n

### SPLITTING TEXT INTO CHUNKS

In [6]:
def chunking(documents, chunk_size=1000,chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    return split_docs

In [7]:
chunks = chunking(all_pdf_documents)

Split 7 documents into 40 chunks


In [8]:
print(chunks)

[Document(metadata={'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '', 'author': 'Md Shamimul Islam; Luis G. Jaimes; Ayesha S. Dina', 'doi': 'https://doi.org/10.48550/arXiv.2604.05458', 'license': 'http://arxiv.org/licenses/nonexclusive-distrib/1.0/', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'MA-IDS: Multi-Agent RAG Framework for IoT Network Intrusion Detection with an Experience Library', 'trapped': '/False', 'arxivid': 'https://arxiv.org/abs/2604.05458v1', 'source': 'data\\pdf_files\\MA-IDS-Multi-Agent RAG Framework for IoT Network Intrusion.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'MA-IDS-Multi-Agent RAG Framework for IoT Network Intrusion.pdf', 'file_type': 'pdf'}, page_content='MA-IDS: Multi-Agent RAG Framework for IoT Network Intrusion\nDetection with an Experience Library\nMd Shamimul Islam 1, Luis G. Jaimes 2 and Ayesha S. Dina 3\n

### EMBEDDING

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Any, Tuple

In [10]:
model_name="all-MiniLM-L6-v2"

In [11]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self,model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        print(f"loading model started{self.model}")
        self.model = SentenceTransformer(self.model_name)
        print(f"model loaded successfully with its dimension{self.model.get_sentence_embedding_dimension()}")
    
    def generate_embedding(self, texts: List[str])-> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")

        print("Embedding Started....")
        embeddings = self.model.encode(texts , show_progress_bar=True)
        print(f"generated embeddings with shape{embeddings.shape}")

        return embeddings
    

emb_man = EmbeddingManager()
        

loading model startedNone


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1172.87it/s]


model loaded successfully with its dimension384


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_24360\1117816385.py:11: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"model loaded successfully with its dimension{self.model.get_sentence_embedding_dimension()}")


In [12]:
texts=[doc.page_content for doc in chunks]

In [13]:
embeddings = emb_man.generate_embedding(texts)

Embedding Started....


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Batches: 100%|██████████| 2/2 [00:05<00:00,  2.62s/it]

generated embeddings with shape(40, 384)


### Vector DB

In [16]:
import numpy as np
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
client = chromadb.PersistentClient(path="data/chroma")
collection = client.get_or_create_collection(name = "pdf_documents", metadata={"hnsw:space": "cosine"})

In [17]:
class vectorStore:
    def __init__(self):
        self.client = chromadb.PersistentClient(path="data/chroma")
        self.collection = self.client.get_or_create_collection(name = "pdf_documents", metadata={"hnsw:space": "cosine"})

        
    def add_document(self, documents:list[any], embeddings:np.ndarray):
        
        ids =[]
        embeddings_list = []
        texts = []
        metadatas = []

        for i,(doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content) 
            metadatas.append(metadata)

            texts.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=texts
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

In [18]:
vector_store = vectorStore()

In [19]:
vector_store.add_document(chunks, embeddings)

Successfully added 40 documents to vector store
Total documents in collection: 40


In [20]:
class RagRetriever:
    def __init__(self, vectorstore: vectorStore, embedding_manager: EmbeddingManager ):
        self.vectorstore = vectorstore
        self.embedding_man = embedding_manager

    def retrieve(self, query: str , top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        ## converting query into embedding

        query_emb = self.embedding_man.generate_embedding([query])[0]

        ## Result

        results = self.vectorstore.collection.query(
            query_embeddings=[query_emb.tolist()],
            n_results=top_k
        )

        retrieved_docs=[]
        if results['documents'] and results['documents'][0]:
            documents = results['documents'][0]
            metadatas = results['metadatas'][0]
            distances = results['distances'][0]
            ids = results['ids'][0]

            for i,(doc_id, document, metadata, distance) in enumerate (zip(ids, documents, metadatas, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id" : doc_id,
                        "content" : document,
                        "metadata" : metadata,
                        "rank" : i+1,
                        "similarity_score" : similarity_score,
                        "distance" : distance}
                    )

            print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
        else:
            print("No documents found")

        return retrieved_docs


OurRagRetriever = RagRetriever(vector_store, emb_man)


In [39]:
OurRagRetriever.retrieve("What is Multi-Agent RAG Framework for IoT Network Intrusion Detection")

Embedding Started....


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.05it/s]

generated embeddings with shape(1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_94c2accf_0',
  'content': 'MA-IDS: Multi-Agent RAG Framework for IoT Network Intrusion\nDetection with an Experience Library\nMd Shamimul Islam 1, Luis G. Jaimes 2 and Ayesha S. Dina 3\nAbstract— Network Intrusion Detection Systems (NIDS) face\nimportant limitations. Signature-based methods are effective\nfor known attack patterns, but they struggle to detect zero-\nday attacks and often miss modified variants of previously\nknown attacks, while many machine learning approaches offer\nlimited interpretability. These challenges become even more\nsevere in IoT environments because of resource constraints\nand heterogeneous protocols. To address these issues, we\npropose MA-IDS, a Multi-Agent Intrusion Detection System\nthat combines Large Language Models (LLMs) with Retrieval\nAugmented Generation (RAG) for reasoning-driven intrusion\ndetection. The proposed framework grounds LLM reasoning\nthrough a persistent, self-building Experience Library. Two\nspecialized agents colla

In [21]:
OurRagRetriever.retrieve("Experience Library Rule Distribution")

Embedding Started....


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.13it/s]

generated embeddings with shape(1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_11b7bf71_29',
  'content': 'the dominant performance gain, while the MA-IDS supports\ncontinual adaptation through rule expansion.\nG. Experience Library Rule Distribution\nTable II summarizes the number of the rules generated per\nclass during the library construction phase. Rule counts re-\nflect class-wise misclassification frequency, with overlapping\ntraffic patterns producing more rules. On NF-BoT-IoT, DDoS\nand DoS generate the highest counts, while on NF-ToN-\nIoT, Injection and XSS dominate due to greater semantic',
  'metadata': {'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1',
   'doi': 'https://doi.org/10.48550/arXiv.2604.05458',
   'source_file': 'MA-IDS-Multi-Agent RAG Framework for IoT Network Intrusion.pdf',
   'page_label': '5',
   'author': 'Md Shamimul Islam; Luis G. Jaimes; Ayesha S. Dina',
   'arxivid': 'https://arxiv.org/abs/2604.05458v1',
   'creationdate': '',
   'trapped': '/False',
   'pr

In [22]:
OurRagRetriever.retrieve("Evaluation Metrics")

Embedding Started....


Batches: 100%|██████████| 1/1 [00:00<00:00, 25.09it/s]

generated embeddings with shape(1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_fb40f38f_22',
  'content': 'with selected volumetric and temporal metrics; (ii) limited\ndiscriminative contribution, particularly for protocol-specific\nor rarely populated fields (e.g., DNS, FTP, ICMP); and (iii)\nincompatibility with privacy-preserving and encrypted traffic\nsettings, where payload-dependent or fine-grained inspection\nfeatures are unreliable or unavailable.\nDuring preprocessing, invalid values are normalized, pro-\ntocol identifiers are mapped to categorical representations,\nand all features are serialized into structured JSON format\nto enable efficient and interpretable LLM-based reasoning.\nC. Evaluation Metrics\nPerformance for all configurations was evaluated using\noverall accuracy along with macro averaged precision, recall,\nand F1 score. The macro averaging technique calculates each\nmetric independently for every individual class before com-\nputing the arithmetic mean across the entire set of classes.\nThis approach assigns equal weight to

### INITIALIZING LLM

In [ ]:
import os
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0.1,max_tokens=1024)

llm.invoke("Hi, how are you doing?") 

AIMessage(content="Hello! I'm doing great, thanks for asking. How can I assist you today?", additional_kwargs={'reasoning_content': 'We need to respond as ChatGPT. No policy violation. Just a friendly greeting.'}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 78, 'total_tokens': 122, 'completion_time': 0.281933661, 'completion_tokens_details': {'reasoning_tokens': 18}, 'prompt_time': 0.00361709, 'prompt_tokens_details': None, 'queue_time': 0.403539128, 'total_time': 0.285550751}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_bb691ea66b', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0c4b0-725b-7682-ba46-0d40bb2b9061-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 78, 'output_tokens': 44, 'total_tokens': 122, 'output_token_details': {'reasoning': 18}})

### PIPELINE (RAG_CONTEXT -> LLM ANSWER)

In [24]:
def simpleResponse(query:str, retriever, llm, top_k):
    """query is the user query"""

    results = retriever.retrieve(query, top_k = top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""

    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [26]:
simpleResponse("what is the performance of MA_IDS on the NF-BoT-IoT and NF-ToN-IoT datasets",OurRagRetriever,llm, top_k=5 )

Embedding Started....


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  9.95it/s]

generated embeddings with shape(1, 384)
Retrieved 5 documents (after filtering)


'**NF‑BoT‑IoT**  \n- Macro\u202fF1‑Score: **≈\u202f89.75\u202f%** (the full MA‑IDS evaluation reports 90\u202f% accuracy, 90\u202f% precision and 90\u202f% recall, yielding a macro\u202fF1 of 89.75\u202f%).  \n\n**NF‑ToN‑IoT**  \n- With the Experience Library active (the “Library‑Only” MA‑IDS configuration) the system attains a **macro\u202fF1 of ≈\u202f85.22\u202f%**.  \n- When the full MA‑IDS pipeline (including ongoing rule induction) is run, the macro\u202fF1 is **≈\u202f81.01\u202f%**.  \n\nThus, MA‑IDS achieves roughly 90\u202f% macro‑F1 on NF‑BoT‑IoT and between 81\u202f%–85\u202f% macro‑F1 on NF‑ToN‑IoT, far surpassing the zero‑shot GPT‑4o baseline (≈\u202f17\u202f% and ≈\u202f5\u202f% respectively).'

In [27]:
def advanceResponse(query:str , retrieval, llm, top_k: int = 5, return_context=False):
    results = retrieval.retrieve(query, top_k=top_k)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    context = "/n/n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page' : doc['metadata'].get('page','unknown'),
        'similarity_score' : doc["similarity_score"],
        'preview' : doc['content'][:300]+'....'
    }for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
    'answer': response.content,
    'sources': sources,
    'confidence': confidence
    }

    if return_context:
        output['context'] = context
    return output

In [30]:
result = advanceResponse("what is the performance of MA_IDS on the NF-BoT-IoT and NF-ToN-IoT datasets",OurRagRetriever,llm, top_k=5, return_context=True )

Embedding Started....


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.55it/s]

generated embeddings with shape(1, 384)


Retrieved 5 documents (after filtering)


In [36]:
result['context'][:300]

'0%\n10%\n20%\n30%\n40%\n50%\n60%\n70%\n80%\n90%\n100%\n0 10,000 20,000 30,000 40,000 50,000\nSamples\nMacro F1 Score\nWith Library Without Library\n(a) NF-BoT-IoT dataset\n0%\n10%\n20%\n30%\n40%\n50%\n60%\n70%\n80%\n90%\n100%\n10,000 20,000 30,000 40,000 50,000\nSamples\nMacro F1 Score\nWith Library Without Library (b) NF-ToN-Io'

In [32]:
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'])

Answer: **NF‑BoT‑IoT** – When the Experience Library is fixed (evaluation stage) MA‑IDS achieves  

- Accuracy: **90 %**  
- Macro‑Precision: **90 %**  
- Macro‑Recall: **90 %**  
- Macro‑F1‑Score: **≈ 89.8 %**  

**NF‑ToN‑IoT** – With the same fixed library, MA‑IDS attains a  

- Macro‑F1‑Score of **≈ 85.2 %**  

(its overall accuracy in the full MA‑IDS run is reported around **81 %**, but the key reported metric for this dataset is the 85.2 % macro‑F1).
Sources: [{'source': 'MA-IDS-Multi-Agent RAG Framework for IoT Network Intrusion.pdf', 'page': 4, 'similarity_score': 0.780608057975769, 'preview': '0%\n10%\n20%\n30%\n40%\n50%\n60%\n70%\n80%\n90%\n100%\n0 10,000 20,000 30,000 40,000 50,000\nSamples\nMacro F1 Score\nWith Library Without Library\n(a) NF-BoT-IoT dataset\n0%\n10%\n20%\n30%\n40%\n50%\n60%\n70%\n80%\n90%\n100%\n10,000 20,000 30,000 40,000 50,000\nSamples\nMacro F1 Score\nWith Library Without Library (b) NF-ToN-Io....'}, {'source': 'MA-IDS-Multi-Agent RAG Framework for IoT 

In [41]:
answer = advanceResponse("why Traditional Network Intrusion Detection Systems is not sufficient?",OurRagRetriever,llm, top_k=5, return_context=True )

Embedding Started....


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.37it/s]

generated embeddings with shape(1, 384)
Retrieved 5 documents (after filtering)


In [45]:
print("Answer:", answer['answer'])
print("Sources:", answer['sources'])
print("Confidence:", answer['confidence'])
print("Context Preview:", answer['context'][:200])

Answer: Traditional Network Intrusion Detection Systems rely mainly on signature‑based detection. Such methods are **reactive**—they can only flag attacks whose signatures are already known—so they miss **zero‑day exploits**, **polymorphic or mutated variants** of known attacks, and consequently suffer high false‑negative rates. In addition, they provide little insight into why traffic is flagged, making them hard to interpret and adapt, especially in resource‑constrained, heterogeneous IoT environments. Hence, signature‑only NIDS are insufficient for modern, dynamic threat landscapes.
Sources: [{'source': 'MA-IDS-Multi-Agent RAG Framework for IoT Network Intrusion.pdf', 'page': 5, 'similarity_score': 0.515844464302063, 'preview': 'automotive embedded networks,”Int. J. Embedded Systems, vol. 10,\nno. 1, pp. 1–12, 2018.\n[17] G. Kim, S. Lee, and S. Kim, “A novel hybrid intrusion detection\nmethod integrating anomaly detection with misuse detection,”Expert\nSystems with Applications, vol

### Advance Rag Pipeline

In [46]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(OurRagRetriever, llm)
result = adv_rag.query("why Traditional Network Intrusion Detection Systems is not sufficient?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Embedding Started....


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.77it/s]

generated embeddings with shape(1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
automotive embedded networks,”Int. J. Embedded Systems, vol. 10,
no. 1, pp. 1–12, 2018.
[17] G. Kim, S. Lee, and S. Kim, “A novel hybrid intrusion detection
method integra

ting anomaly detection with misuse detection,”Expert
Systems with Applications, vol. 41, no. 4, pp. 1690–1700, 2014.
[18] K. Bajaj and A. Arora, “Dimension reduction in intrusion detection
features using discriminative machine learning approach,”Int. J.
Computer Science Issues, vol. 10, no. 4, p. 324, 2013.
[19] W. Li, P. Yi, Y . Wu, L. Pan, and J. Li, “A new intrusion detection sys-
tem based on knn classification algorithm in wireless sensor network,”
J. Electrical and Computer Engineering, vol. 2014.
[20] A. Guezzaz, S. Benkirane, M. Azrour, and S. Khurram, “A reliable net-
work intrusion detection approach using decision tree with enhanced
data quality,”Security and Communication Networks, vol. 2021, 2021.
[21] M. Mohammadi et al., “A comprehensive survey and taxonomy of

detection system,”Materials Today: Proceedings, vol. 47, pp. 139–
143, 2021.
[12] M. Y . AlYousef and N. T. Abdelmajeed, “Dynamically detecting
security threats and updating a signature-based intrusion detection
s

In [47]:
result = adv_rag.query("who is muhammad minhal?", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Embedding Started....


Batches: 100%|██████████| 1/1 [00:00<00:00, 16.95it/s]

generated embeddings with shape(1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
data. However, these models typically operate as “black
1Md Shamimul Islam is with the Department of Computer
Science, Florida Polytechnic University, Lakeland, Florida
mi

slam3051@floridapoly.edu
2Luis G. Jaimes is with the Department of Computer
Science, Florida Polytechnic University, Lakeland, Florida
ljaimes@floridapoly.edu
3Ayesha S. Dina is with the Department of Computer
Science, Florida Polytechnic University, Lakeland, Florida
adina@floridapoly.edu
This work has been submitted to the IEEE for possible publication.
Copyright may be transferred without notice, after which this version may
no longer be accessible.
boxes,” providing high classification accuracy at the expense
of interpretability. This is a critical shortcoming in security-
critical environments where forensic justification is required
for incident response. Furthermore, their reliance on static
training distributions makes them brittle against evolving
threats, often necessitating computationally expensive re-

MA-IDS: Multi-Agent RAG Framework for IoT Network Intrusion
Detection with an Experience Library
Md Shamimul Islam 1, Luis G. Jaimes 2 and Ayesha S. Dina 3
Abstract— Network